In [1]:
import torch
print("GPU available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU - check Runtime settings")

GPU available: True
Tesla T4


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

DATA = "/content/drive/MyDrive/fake-news/"

def load(path):
    df = pd.read_csv(path).dropna(subset=["text"])
    df["content"] = df["title"].fillna("") + ". " + df["text"]
    df["y"] = (df["label"] == "REAL").astype(int)      # 1 = REAL, 0 = FAKE
    return df[["content", "y"]].reset_index(drop=True)

orig = load(DATA + "fake_or_real_news.csv")
wel = load(DATA + "welfake_clean.csv")

# Same random_state and stratify as cross_dataset_test.py -> identical splits
o_tr, o_te = train_test_split(orig, test_size=0.2, random_state=7, stratify=orig["y"])
w_tr, w_te = train_test_split(wel, test_size=0.2, random_state=7, stratify=wel["y"])

print(f"Original: {len(o_tr)} train / {len(o_te)} test")
print(f"WELFake:  {len(w_tr)} train / {len(w_te)} test")

Original: 5068 train / 1267 test
WELFake:  45013 train / 11254 test


In [5]:
import time
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

MODEL = "distilbert-base-uncased"
MAX_LEN = 256          # DistilBERT reads the first ~256 tokens (~180 words) of each article
BATCH = 32
device = "cuda"

tok = AutoTokenizer.from_pretrained(MODEL)


class NewsDataset(Dataset):
    def __init__(self, df):
        self.texts = df["content"].tolist()
        self.labels = df["y"].tolist()
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return self.texts[i], self.labels[i]


def collate(batch):
    texts, labels = zip(*batch)
    enc = tok(list(texts), truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(labels)
    return enc


def train_model(train_df, epochs, lr=2e-5):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).to(device)
    loader = DataLoader(NewsDataset(train_df), batch_size=BATCH, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = epochs * len(loader)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda")

    model.train()
    for ep in range(epochs):
        start, running = time.time(), 0
        for i, batch in enumerate(loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            running += loss.item()
            if i % 100 == 0:
                print(f"  epoch {ep + 1}  step {i}/{len(loader)}  loss {loss.item():.3f}")
        print(f"Epoch {ep + 1} done - avg loss {running / len(loader):.3f} ({(time.time() - start) / 60:.1f} min)")
    return model


@torch.no_grad()
def evaluate(model, df, name):
    model.eval()
    preds = []
    loader = DataLoader(NewsDataset(df), batch_size=64, collate_fn=collate)
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        with torch.autocast("cuda", dtype=torch.float16):
            preds += model(**batch).logits.argmax(-1).tolist()
    acc = accuracy_score(df["y"], preds)
    f1 = f1_score(df["y"], preds, average="macro")
    print(f"{name}: accuracy {acc:.3f} | macro F1 {f1:.3f}")
    return acc, f1

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
results = []

print("Experiment 1: train on ORIGINAL")
model_o = train_model(o_tr, epochs=2)
results.append(("Original", "Original (same)", *evaluate(model_o, o_te, "Original test")))
results.append(("Original", "WELFake (new)", *evaluate(model_o, wel, "WELFake (all)")))

Experiment 1: train on ORIGINAL


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1  step 0/159  loss 0.684
  epoch 1  step 100/159  loss 0.111
Epoch 1 done - avg loss 0.279 (1.1 min)
  epoch 2  step 0/159  loss 0.157
  epoch 2  step 100/159  loss 0.016
Epoch 2 done - avg loss 0.066 (0.6 min)
Original test: accuracy 0.962 | macro F1 0.962
WELFake (all): accuracy 0.824 | macro F1 0.816


In [7]:
print("Experiment 2: train on WELFAKE")
del model_o; torch.cuda.empty_cache()          # free GPU memory from experiment 1
model_w = train_model(w_tr, epochs=1)          # 45k articles: one pass is enough
results.append(("WELFake", "WELFake (same)", *evaluate(model_w, w_te, "WELFake test")))
results.append(("WELFake", "Original (new)", *evaluate(model_w, orig, "Original (all)")))

Experiment 2: train on WELFAKE


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch 1  step 0/1407  loss 0.683
  epoch 1  step 100/1407  loss 0.210
  epoch 1  step 200/1407  loss 0.226
  epoch 1  step 300/1407  loss 0.026
  epoch 1  step 400/1407  loss 0.006
  epoch 1  step 500/1407  loss 0.009
  epoch 1  step 600/1407  loss 0.002
  epoch 1  step 700/1407  loss 0.029
  epoch 1  step 800/1407  loss 0.010
  epoch 1  step 900/1407  loss 0.142
  epoch 1  step 1000/1407  loss 0.004
  epoch 1  step 1100/1407  loss 0.003
  epoch 1  step 1200/1407  loss 0.016
  epoch 1  step 1300/1407  loss 0.174
  epoch 1  step 1400/1407  loss 0.009
Epoch 1 done - avg loss 0.069 (5.6 min)
WELFake test: accuracy 0.995 | macro F1 0.995
Original (all): accuracy 0.700 | macro F1 0.673


In [8]:
table = pd.DataFrame(results, columns=["Trained on", "Tested on", "Accuracy", "Macro F1"])
print(table.to_string(index=False, float_format="%.3f"))
table.to_csv(DATA + "distilbert_results.csv", index=False)
print("\nSaved to Google Drive: fake-news/distilbert_results.csv")

Trained on       Tested on  Accuracy  Macro F1
  Original Original (same)     0.962     0.962
  Original   WELFake (new)     0.824     0.816
   WELFake  WELFake (same)     0.995     0.995
   WELFake  Original (new)     0.700     0.673

Saved to Google Drive: fake-news/distilbert_results.csv
